# Tigrinya Word Embeddings from Scratch

## Skip-gram Neural Network in Pure Python

**Goal:** train word embeddings for Tigrinya with a skip-gram model implemented
by hand: forward pass, stable softmax, loss, gradients and SGD.

**Constraints:** only Python lists, loops, `math` and `random`. No NumPy, no
automatic differentiation, no neural-network libraries and no pretrained embeddings.

### Pipeline

Raw corpus → Cleaning → Tokenization → Vocabulary → Integer IDs →
Center-context pairs → Forward pass → Stable softmax + loss → Manual gradients →
SGD training → Embedding matrix E → Cosine similarity / nearest neighbors

### Notation (row vectors)

`h = E[i]` (1×d), `s = hU` (1×V), `p = softmax(s)`,
`L = (m − s[t]) + ln Σ exp(s[j] − m)`, `e = p − onehot(t)`,
`grad_U = hᵀe`, `grad_h = eUᵀ`, with E of shape V×d and U of shape d×V.

### Notebook sections

1. Setup and settings
2. Data preparation (vocabulary and training pairs)
3. Implementation checks (the required tests)
4. Training (J before training and after each epoch)
5. Nearest neighbors
6. Optional: save/reload, window or dimension comparison
7. Conclusions

The model code is in `src/`. This notebook only runs the experiment and
shows the results.

### Imports and project setup

In [1]:
from src.preprocessing import Preprocessor
from src.model import SkipGram
from src.training import Trainer
from src.embeddings import Embeddings


## Preprocessor

### Load and Prepare Preprocessor

In [2]:
SEED, DIM, WINDOW, LR, EPOCHS = 42, 10, 3, 0.05, 50

pre = Preprocessor(window=WINDOW)

sents = pre.load_sentences("data/raw/tigrinya_corpus.txt")

print("Number of sentences:", len(sents))
print("First 5 sentences", sents[:5])

Number of sentences: 200
First 5 sentences [['እቲ', 'ተማሃራይ', 'ንግሆ', 'ናብ', 'ቤት', 'ትምህርቲ', 'ከደ'], ['እታ', 'ተማሃሪት', 'ሓድሽ', 'መጽሓፍ', 'ኣንበበት'], ['እቲ', 'መምህር', 'ንተማሃሮ', 'ሓድሽ', 'ትምህርቲ', 'ኣመሃሮም'], ['እቶም', 'ተማሃሮ', 'ኣብ', 'ክፍሊ', 'ብጽሞና', 'ሰምዑ'], ['ኣነ', 'ሎሚ', 'ናይ', 'ገዛ', 'ስራሐይ', 'ወዲአ']]


### Build Vocabulary

In [3]:
pre.build_vocab(sents)

print("Vocabulary size:", len(pre.id2word))

print("\nFirst 10 vocabulary words:")
for i, word in enumerate(pre.id2word[:10]):
    print(i, word)

Vocabulary size: 495

First 10 vocabulary words:
0 ሃበት
1 ሃበቶ
2 ሃቦ
3 ህዝባዊ
4 ህይወት
5 ሆስፒታል
6 ሎሚ
7 ሒዙ
8 ሒዛ
9 ሓለፈ


### Generate Context Pairs

In [4]:
pairs = pre.make_pairs(sents)

print("Number of training pairs:", len(pairs))

print("\nFirst 10 training pairs:")
for pair in pairs[:10]:
    center_id, context_id = pair
    print(pre.id2word[center_id], "->", pre.id2word[context_id])

Number of training pairs: 4392

First 10 training pairs:
እቲ -> ተማሃራይ
እቲ -> ንግሆ
እቲ -> ናብ
ተማሃራይ -> እቲ
ተማሃራይ -> ንግሆ
ተማሃራይ -> ናብ
ተማሃራይ -> ቤት
ንግሆ -> እቲ
ንግሆ -> ተማሃራይ
ንግሆ -> ናብ


## Create Skip-Gram Model

In [5]:
model = SkipGram(len(pre.id2word), DIM, seed=SEED)

print("Skip-gram model created.")
print("Vocabulary size:", len(pre.id2word))
print("Embedding dimension:", DIM)

Skip-gram model created.
Vocabulary size: 495
Embedding dimension: 10


## Train Model
### Train Skip-Gram Model

In [6]:
trainer = Trainer(model, lr=LR, epochs=EPOCHS, seed=SEED)
history = trainer.fit(pairs)  

before training : J = 6.2048
epoch   1       : J = 6.1976
epoch   2       : J = 6.1352
epoch   3       : J = 5.8658
epoch   4       : J = 5.6097
epoch   5       : J = 5.4066
epoch   6       : J = 5.2170
epoch   7       : J = 5.0520
epoch   8       : J = 4.8825
epoch   9       : J = 4.7324
epoch  10       : J = 4.5828
epoch  11       : J = 4.4442
epoch  12       : J = 4.3102
epoch  13       : J = 4.1851
epoch  14       : J = 4.0805
epoch  15       : J = 3.9734
epoch  16       : J = 3.8834
epoch  17       : J = 3.8073
epoch  18       : J = 3.7236
epoch  19       : J = 3.6631
epoch  20       : J = 3.5973
epoch  21       : J = 3.5416
epoch  22       : J = 3.5036
epoch  23       : J = 3.4741
epoch  24       : J = 3.4319
epoch  25       : J = 3.4014
epoch  26       : J = 3.3748
epoch  27       : J = 3.3554
epoch  28       : J = 3.3591
epoch  29       : J = 3.3160
epoch  30       : J = 3.2867
epoch  31       : J = 3.2683
epoch  32       : J = 3.2622
epoch  33       : J = 3.2489
epoch  34     

In [7]:
import random

def numerical_check(model, i, t, eps=1e-5, n_u=30, seed=0):
    """Max |analytic - numeric| over all of E[i] and a random sample of U entries."""
    rnd = random.Random(seed)
    gU, gh, _ = model.gradients(i, t)

    def num(M, r, c):                                   # [L(x+eps) - L(x-eps)] / (2 eps)
        orig = M[r][c]
        M[r][c] = orig + eps; up = model.pair_loss(i, t)
        M[r][c] = orig - eps; down = model.pair_loss(i, t)
        M[r][c] = orig                                  # restore exactly
        return (up - down) / (2 * eps)

    errs = [abs(num(model.E, i, k) - gh[k]) for k in range(model.d)]
    for _ in range(n_u):
        k, j = rnd.randrange(model.d), rnd.randrange(model.V)
        errs.append(abs(num(model.U, k, j) - gU[k][j]))
    return max(errs)

for (i, t) in random.Random(1).sample(pairs, 3):
    err = numerical_check(model, i, t)
    print(pre.id2word[i], "->", pre.id2word[t], " max error =", f"{err:.2e}", " OK" if err < 1e-6 else " CHECK")

ቆሪርን -> እዩ  max error = 8.99e-11  OK
ኣብ -> የንብብ  max error = 1.68e-10  OK
ብጥንቃቐ -> ሰባት  max error = 8.84e-11  OK


## Query Learned Embeddings
### Query Nearest Neighbors

In [8]:
emb = Embeddings(model.E, pre.word2id, pre.id2word)
for w in ["ተማሃሮ", "ገዛ", "ዕዳጋ", "ምግቢ", "ዝናብ", "ከተማ", "ጽቡቕ", "ብሓባር", "ኮምፒተር", "መኪና", "ሰላም", "ሓደሽቲ", "ኣብዚ"]:
    res = emb.nearest(w, k=3)
    print(w, res if res else "not in vocabulary")       



ተማሃሮ [('ማሽን_ለርኒግ', 0.8011), ('ነበርቲ', 0.7723), ('ወረዱ', 0.772)]
ገዛ [('መልኦ', 0.9322), ('ማዕጾ', 0.8931), ('ወዲአ', 0.8553)]
ዕዳጋ [('ብታክሲ', 0.8318), ('ብእግሮም', 0.8147), ('ክዕድግ', 0.8059)]
ምግቢ [('ጥዑም', 0.937), ('ንስድራቤት', 0.8125), ('ጠረጴዛና', 0.764)]
ዝናብ [('ድሕሪ', 0.8809), ('ዘነበ', 0.8685), ('ካብ', 0.8395)]
ከተማ [('እታ', 0.8731), ('ጓል', 0.8137), ('ኣለዋ', 0.7808)]
ጽቡቕ [('ጉዕዞ', 0.8687), ('ንኣካላዊ', 0.8317), ('ብቕዓት', 0.8303)]
ብሓባር [('ኣዕሩኽትና', 0.7691), ('ድራር', 0.7611), ('ንሕና', 0.7176)]
ኮምፒተር not in vocabulary
መኪና [('ነቲ', 0.8569), ('ጐዓዘ', 0.8518), ('ኣሽከርካሪ', 0.8234)]
ሰላም not in vocabulary
ሓደሽቲ not in vocabulary
ኣብዚ not in vocabulary


### Conclusion paragraph you can use

The final model uses seed 42, embedding dimension 10, window 3, learning rate 0.05, 50 epochs and min_count 1. Mean training loss fell from 6.20 (about ln V, a uniform guess) to 3.16 and plateaued from about epoch 40. Window 3 produced more topically coherent neighbors than window 2; dimension 5 was too coarse, while dimensions 10 and 20 gave similar quality, so 10 was chosen for its lower cost and smaller risk of memorizing a small corpus. A learning rate of 0.1 oscillated near the end of training and reached a higher loss (3.33) than 0.05 (3.16) on the same pairs. Raising min_count to 2 removed rare-word artifacts but also removed informative specific neighbors, so it was kept at 1. Neighbors mostly reflect words that co-occur in the same sentences rather than true synonyms, and rare words (such as ማሽን_ለርኒግ) still get unreliable neighbors. With a corpus of about 200 sentences, one seed and no held-out data, these differences are suggestive rather than conclusive, and low training loss alone does not establish semantic quality.

### Save and Reload Model

In [9]:
model.save("models/tigrinya.json", pre.id2word,
           meta={"window": WINDOW, "lr": LR, "epochs": EPOCHS, "loss_history": history})

model2, vocab2 = SkipGram.load("models/tigrinya.json")
print(model2.meta)

{'seed': 42, 'vocab_size': 495, 'dim': 10, 'window': 3, 'lr': 0.05, 'epochs': 50, 'loss_history': [6.204750117581964, 6.197619719460254, 6.135194036278042, 5.865816039944726, 5.609720098505944, 5.406647894165379, 5.2169555599224005, 5.05195185040013, 4.882517218688456, 4.732408417591713, 4.582752075533182, 4.444159175391888, 4.310217917973495, 4.185087813113009, 4.08045900327147, 3.9733768117863635, 3.8834290370036615, 3.8072552521151835, 3.7235552043727527, 3.6631462636629233, 3.5972960466033492, 3.541649902689439, 3.503578463489731, 3.4740897698359547, 3.4318652366041635, 3.4013984687130976, 3.374769857809953, 3.355385632371211, 3.359081306664419, 3.31603980370969, 3.2866633057422496, 3.268262290075209, 3.2622061427051, 3.248906585547273, 3.2485509390023544, 3.2472928094036027, 3.2323458885091862, 3.212630186290078, 3.206702320111545, 3.1909854258585284, 3.191791558877041, 3.198529720682382, 3.1738460728669686, 3.1870687023821076, 3.1679456493932125, 3.166612474925241, 3.168497415696